# Feature Engineering

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler

In [64]:
## Load data
df = pd.read_csv("..\\data\\processed\\cleaned_telco_customer_churn.csv")
print(df.shape)
df.head()

(7021, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [65]:
## Split features and target
X, y = df.iloc[:, :-1], df.iloc[:,-1]
print(X.shape, y.shape)

(7021, 19) (7021,)


In [66]:
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [67]:
y.head()

0     No
1     No
2    Yes
3     No
4    Yes
Name: Churn, dtype: object

In [68]:
## Split the dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5616, 19)
(1405, 19)
(5616,)
(1405,)


In [69]:
## Categorical Featues
cat_features = X_train.select_dtypes(include=['object']).columns
cat_features

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [70]:
## Numerical features
num_features = X_train.select_dtypes(exclude='object').columns
num_features

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [71]:
len(cat_features) + len(num_features)

19

In [72]:
## unique values in cat_features
unique_cat_list= []
for feature in cat_features:
    unique_cat_series = X_train[feature].value_counts()
    unique_cat_list.append({'FeatureName': feature, 'UniqueValues' : unique_cat_series.index.values, 'Numbers':unique_cat_series.values})

unique_cat_X_train = pd.DataFrame(unique_cat_list)
unique_cat_X_train

,FeatureName,UniqueValues,Numbers
0,gender,"[Male, Female]","[2854, 2762]"
1,Partner,"[No, Yes]","[2915, 2701]"
2,Dependents,"[No, Yes]","[3938, 1678]"
3,PhoneService,"[Yes, No]","[5070, 546]"
4,MultipleLines,"[No, Yes, No phone service]","[2703, 2367, 546]"
5,InternetService,"[Fiber optic, DSL, No]","[2458, 1949, 1209]"
6,OnlineSecurity,"[No, Yes, No internet service]","[2783, 1624, 1209]"
7,OnlineBackup,"[No, Yes, No internet service]","[2498, 1909, 1209]"
8,DeviceProtection,"[No, Yes, No internet service]","[2480, 1927, 1209]"
9,TechSupport,"[No, Yes, No internet service]","[2774, 1633, 1209]"


#### Comment:
The above table shows that most services categorical features like (MultipleLines, InternetService, etc) has 3 unique categories but there is actually 2 meaningful categories "yes", and "No" because "No internet service" indicates "No".

So, we can replace "Yes" by 1 and "No" or "No internet service" by 0.

### Encoding

In [73]:
## Encode binary features
binary_feat = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

# Encode binary cat cols manually except 'gender'
X_train[binary_feat[1:]]= X_train[binary_feat[1:]].map(lambda x: 1 if x=='Yes' else 0)
X_test[binary_feat[1:]]= X_test[binary_feat[1:]].map(lambda x: 1 if x=='Yes' else 0)

# Encode 'gender' with OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
X_train['gender']=ohe.fit_transform(X_train[['gender']])
X_test['gender']=ohe.transform(X_test[['gender']])

X_train.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3269,1.0,0,0,0,40,1,1,Fiber optic,0,0,0,0,1,0,Month-to-month,1,Electronic check,84.90,3369.05
6252,1.0,1,1,0,27,1,1,Fiber optic,1,0,1,0,0,0,One year,1,Electronic check,85.90,2220.10
1323,1.0,0,0,0,31,1,0,No,0,0,0,0,0,0,Two year,0,Bank transfer (automatic),20.40,609.10
5190,0.0,0,1,1,14,1,0,DSL,1,1,0,1,0,0,Month-to-month,0,Bank transfer (automatic),59.45,780.85
4071,1.0,0,0,0,7,1,1,Fiber optic,0,0,0,0,0,0,Month-to-month,0,Electronic check,73.60,520.00


In [74]:
## Encode multiple categorical features
non_binary_cats = ['InternetService', 'Contract', 'PaymentMethod']

## OneHotEncoding
ohe = OneHotEncoder(sparse_output=False, drop='first').set_output(transform="pandas")

## Encode
ohe_encoded_X_train = ohe.fit_transform(X_train[non_binary_cats])
ohe_encoded_X_test = ohe.transform(X_test[non_binary_cats])

ohe_encoded_X_train

,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3269,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6252,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1323,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5190,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4071,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...
3772,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5191,1.0,0.0,0.0,1.0,1.0,0.0,0.0
5226,0.0,1.0,0.0,1.0,1.0,0.0,0.0
5390,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [75]:
## Added to the original dataframe
X_train = X_train.drop(columns=non_binary_cats).join(ohe_encoded_X_train)
X_test = X_test.drop(columns=non_binary_cats).join(ohe_encoded_X_test)

print(X_train.shape, X_test.shape)
X_train.head()

(5616, 23) (1405, 23)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3269,1.0,0,0,0,40,1,1,0,0,0,...,1,84.90,3369.05,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6252,1.0,1,1,0,27,1,1,1,0,1,...,1,85.90,2220.10,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1323,1.0,0,0,0,31,1,0,0,0,0,...,0,20.40,609.10,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5190,0.0,0,1,1,14,1,0,1,1,0,...,0,59.45,780.85,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4071,1.0,0,0,0,7,1,1,0,0,0,...,0,73.60,520.00,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [76]:
type(y_train)

pandas.core.series.Series

In [77]:
## Encode the target col
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train

array([0, 0, 0, ..., 0, 1, 0], shape=(5616,))

### Feature Scaling

In [78]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3269,1.0,0,0,0,40,1,1,0,0,0,...,1,84.90,3369.05,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6252,1.0,1,1,0,27,1,1,1,0,1,...,1,85.90,2220.10,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1323,1.0,0,0,0,31,1,0,0,0,0,...,0,20.40,609.10,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5190,0.0,0,1,1,14,1,0,1,1,0,...,0,59.45,780.85,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4071,1.0,0,0,0,7,1,1,0,0,0,...,0,73.60,520.00,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [79]:
## Scaling except target
scaler = StandardScaler()
for col in num_features[1:]:
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_test[col] = scaler.transform(X_test[[col]])

X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaperlessBilling,MonthlyCharges,TotalCharges,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3269,1.0,0,0,0,0.321940,1,1,0,0,0,...,1,0.674759,0.495749,1.0,0.0,0.0,0.0,0.0,1.0,0.0
6252,1.0,1,1,0,-0.208976,1,1,1,0,1,...,1,0.708136,-0.015291,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1323,1.0,0,0,0,-0.045617,1,0,0,0,0,...,0,-1.478076,-0.731845,0.0,1.0,0.0,1.0,0.0,0.0,0.0
5190,0.0,0,1,1,-0.739893,1,0,1,1,0,...,0,-0.174693,-0.655453,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4071,1.0,0,0,0,-1.025771,1,1,0,0,0,...,0,0.297596,-0.771476,1.0,0.0,0.0,0.0,0.0,1.0,0.0


#### Comment:
All features have been successfully converted to required format for feeding ML algorithms. Now we can export this data and can use for machine learning.

In [80]:
y_train

array([0, 0, 0, ..., 0, 1, 0], shape=(5616,))

In [81]:
## Convert y_train and y_test data from numpy array to pandas series format
y_train_series = pd.Series(y_train, name='Churn')
y_test_series = pd.Series(y_test, name='Churn')
y_train_series.head()

0    0
1    0
2    0
3    0
4    1
Name: Churn, dtype: int64

### Export the cleaned and transformed data

In [82]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_series.to_csv('../data/processed/y_train.csv', index=False)
y_test_series.to_csv('../data/processed/y_test.csv', index=False)

<br/>

---
### 👨‍💻 Author Information
**Name:** [Amaresh Maity]  
**Date:** 2026-01-18  
**Role:** [Data Scientist | AI Engineer]



#### Let's Connect!

If you have questions about this analysis or would like to collaborate, feel free to reach out:

* **LinkedIn:** [LinkedIn](https://www.linkedin.com/in/amareshmaity/)
* **GitHub:** [@amareshmaity](https://github.com/amareshmaity)
* **Email:** [contacttoamaresh@gmail.com](mailto:contacttoamaresh@gmail.com)
